# Election Prediction Project
## Notebook 03 — Model Training, Comparison and Selection

### Notebook Objectives
This notebook trains and benchmarks several models to predict the following four voting blocs:

Right_pct
Center_Left_pct
Haredi_pct
Arab_pct

### Workflow stages:
*   Chronological Splits: architecture selection on Knesset 24 (training on Knessets 21–23); a single final test on Knesset 25 (training on Knessets 21–24).
*   Baseline Model Training: Evaluation of Linear Regression, Random Forest, and XGBoost models, plus a small neural-network baseline.
*   Feature Contribution: Evaluating the contribution of locality type variables to model performance.
*   Normalization: Normalizing predictions to ensure they sum to 100%.
*   Compositional Analysis: Training a CLR (Centered Log-Ratio) model for compositional data.
*   Segmented Modeling: Training a split CLR model based on the type_Arab/Non-Jewish category.
*   Model Finalization: retraining the selected architecture on all pre-test elections and evaluating it once on the held-out test election.

### input

- `processed_data/modeling_dataset_selected_features.csv`
- `processed_data/selected_demographic_features.json`

### output

- `reports/model_comparison.csv`
- `reports/model_comparison_by_target.csv`
- `processed_data/validation_predictions_selected_model.csv`
- `models/final_model_bundle.joblib`
- `reports/modeling_summary.json`


## 1. Imports and project configuration

In [1]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)

RANDOM_STATE = 99
SELECTION_VALIDATION_ELECTION = "Knesset_24"
TEST_ELECTION = "Knesset_25"

TARGET_COLUMNS = [
    "Right_pct",
    "Center_Left_pct",
    "Haredi_pct",
    "Arab_pct",
]

BASE_CLASSIFICATION_FEATURES = [
    "type_Arab/Non-Jewish",
    "type_Cities",
    "type_Kibbutzim",
    "type_Moshavim",
    "type_other",
]

DRUZE_FEATURE = "type_Druze_Majority"

print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
# Repository-relative configuration (run notebooks from the notebooks/ folder):
# all paths stay relative, so the project runs on any machine straight after cloning.
REPO_ROOT = Path('..')

PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = REPO_ROOT / 'reports'
MODELS_DIR = REPO_ROOT / 'models'

INPUT_PATH = PROCESSED_DIR / 'modeling_dataset_selected_features.csv'
SELECTED_FEATURES_PATH = PROCESSED_DIR / 'selected_demographic_features.json'

MODEL_COMPARISON_PATH = REPORTS_DIR / 'model_comparison.csv'
TARGET_COMPARISON_PATH = REPORTS_DIR / 'model_comparison_by_target.csv'
VALIDATION_PREDICTIONS_PATH = (
    PROCESSED_DIR / 'validation_predictions_selected_model.csv'
)
FINAL_MODEL_BUNDLE_PATH = MODELS_DIR / 'final_model_bundle.joblib'
MODELING_SUMMARY_PATH = REPORTS_DIR / 'modeling_summary.json'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Input dataset: {INPUT_PATH}')
print(f'Selected features: {SELECTED_FEATURES_PATH}')


Input dataset: ..\data\processed\modeling_dataset_selected_features.csv
Selected features: ..\data\processed\selected_demographic_features.json


## 2. Load and validate the modeling dataset

In [3]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Modeling dataset not found: {INPUT_PATH}. "
        "Run Notebook 02 first."
    )

if not SELECTED_FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"Selected-features file not found: {SELECTED_FEATURES_PATH}. "
        "Run Notebook 02 first."
    )

df = pd.read_csv(INPUT_PATH, low_memory=False)

with open(SELECTED_FEATURES_PATH, "r", encoding="utf-8") as file:
    feature_payload = json.load(file)

demographic_features = feature_payload["selected_demographic_features"]

required_columns = [
    "locality_symbol",
    "year",
    "target_election",
    *TARGET_COLUMNS,
    *demographic_features,
    *BASE_CLASSIFICATION_FEATURES,
    DRUZE_FEATURE,
]

missing_columns = [
    column for column in required_columns if column not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df["locality_symbol"] = (
    pd.to_numeric(df["locality_symbol"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

df["year"] = pd.to_numeric(df["year"], errors="coerce")

numeric_columns = list(dict.fromkeys(
    demographic_features
    + BASE_CLASSIFICATION_FEATURES
    + [DRUZE_FEATURE]
    + TARGET_COLUMNS
))

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.replace([np.inf, -np.inf], np.nan)

if df[TARGET_COLUMNS].isna().any().any():
    raise ValueError("Missing target values were found.")

if not np.allclose(df[TARGET_COLUMNS].sum(axis=1), 100, atol=1e-6):
    raise ValueError("Target percentages do not sum to 100%.")

if df.duplicated(
    subset=["locality_symbol", "target_election"]
).any():
    raise ValueError("Duplicate locality-election rows were found.")

print(f"Dataset shape: {df.shape}")
print(f"Selected demographic features: {len(demographic_features)}")
print("Input validation completed successfully.")

display(df.head())


Dataset shape: (1124, 44)
Selected demographic features: 28
Input validation completed successfully.


,locality_symbol,שם יישוב,year,target_election,valid_votes,Other_raw_pct,Right_pct,Center_Left_pct,Haredi_pct,Arab_pct,אחוז עולי 1990+ מסך האוכלוסייה,ערבים (אחוזים מתוך כלל אוכלוסיית הישראלים),gini_wage_2019,עובדי הוראה ממוצע שעות עבודה שבועיות לתלמיד,"אחוז המבוטחים בקופ""ח מתוך סך כל המבוטחים כללית",avg_monthly_wage,גיל ממוצע של מקבלי דמי אבטלה (לא כולל חיילים),השכלה גבוהה אחוז סטודנטים מתוך אוכלוסיית בני 25-20,gender_wage_ratio,"צפיפות אוכלוסייה לקמ''ר ביישובים שמנו 5,000 תושבים ויותר","ריבוי טבעי ל-1,000 תושבים",אחוז באוכלוסייה בסוף השנה בני 4-0,אחוז באוכלוסייה בסוף השנה בני 44-30,"נישאים שיעור ל-1,000 תושבים לא נשואים בני 15 ומעלה",השכלה גבוהה אחוז בעלי תארים מישראל מתוך אוכלוסיית בני 55-35,"מספר ילדים שבגינם שולמו קצבאות במשפחות עם 5 ילדים ויותר ל-1,000 תושבים",אחוז מקבלי השלמת הכנסה מבין מקבלי קצבאות זקנה ושאירים,אחוז זכאים לתעודת בגרות שעמדו בדרישות הסף של האוניברסיטאות מבין תלמידי כיתות יב,עובדי הוראה אחוז בעלי דרגת שכר תואר שני ומעלה,אחוז זכאים לתעודת בגרות מבין תלמידי כיתות יב,"יחס תלות (ל-1,000 תושבים בלתי תלויים)",עובדי הוראה ממוצע שעות עבודה לשבוע,אחוז באוכלוסייה בסוף השנה בני 59-45,מקבלי דמי אבטלה (ממוצע חודשי) גברים,יהודים (אחוזים מתוך יהודים ואחרים),מוסלמים (אחוזים מתוך האוכלוסייה הערבית),יהודים ואחרים (אחוזים מתוך כלל אוכלוסיית הישראלים),נכנסים מיישובים אחרים גיל 64-30,type_Arab/Non-Jewish,type_Cities,type_Kibbutzim,type_Moshavim,type_other,type_Druze_Majority
0,1015,מבשרת ציון,2019,Knesset_21,13846,0.397227,48.198100,47.712276,3.901095,0.188529,7.10,0.5,0.47,2.78,44.9,12147.890000,42.00,14.16,1.395904,4137.3,10.05,7.80,18.80,23.9,42.52,29.341914,4.08,70.10,45.54,80.73,901.1,30.10,16.71,84.00,97.1,93.0,99.5,504.0,0,1,0,0,0,0
1,1015,מבשרת ציון,2019,Knesset_22,13539,0.406234,42.739543,50.927025,6.088698,0.244735,7.10,0.5,0.47,2.78,44.9,12147.890000,42.00,14.16,1.395904,4137.3,10.05,7.80,18.80,23.9,42.52,29.341914,4.08,70.10,45.54,80.73,901.1,30.10,16.71,84.00,97.1,93.0,99.5,504.0,0,1,0,0,0,0
2,1015,מבשרת ציון,2020,Knesset_23,13559,0.103252,45.330380,47.818383,6.253230,0.598007,7.30,0.5,0.47,2.79,45.1,13294.045859,38.12,17.47,1.520429,3850.5,10.72,7.87,17.88,27.7,43.90,35.385821,3.96,71.34,25.68,83.80,868.0,30.50,16.76,694.00,96.7,93.0,99.7,435.0,0,1,0,0,0,0
3,1015,מבשרת ציון,2021,Knesset_24,13331,1.162703,48.337887,46.053430,5.380996,0.227687,7.39,0.5,0.47,2.92,45.1,14557.820000,39.00,17.00,1.489082,3953.0,11.71,8.26,18.02,18.9,44.97,39.209397,3.97,72.59,49.29,83.13,893.0,31.10,16.47,433.00,96.7,93.0,99.6,694.0,0,1,0,0,0,0
4,1015,מבשרת ציון,2022,Knesset_25,14051,1.039072,45.163610,48.212873,6.220784,0.402733,7.60,NaN,0.47,2.88,44.8,15201.840000,38.82,15.60,1.533569,4039.2,9.91,8.10,18.22,29.0,45.74,40.216928,4.37,77.03,49.04,83.75,904.0,31.25,16.27,72.67,96.6,NaN,99.6,539.0,0,1,0,0,0,0


## 3. Chronological training-validation split

In [4]:
# selection and test are two separate chronological splits.
selection_validation_mask = df['target_election'].eq(SELECTION_VALIDATION_ELECTION)
test_mask = df['target_election'].eq(TEST_ELECTION)

selection_train_df = df.loc[~selection_validation_mask & ~test_mask].copy()
selection_validation_df = df.loc[selection_validation_mask].copy()
final_train_df = df.loc[~test_mask].copy()
test_df = df.loc[test_mask].copy()

for subset_name, subset in {
    'selection-training (Knesset 21-23)': selection_train_df,
    'selection-validation (Knesset 24)': selection_validation_df,
    'final-training (Knesset 21-24)': final_train_df,
    'test (Knesset 25)': test_df,
}.items():
    if subset.empty:
        raise ValueError(f'The {subset_name} dataset is empty.')

y_selection_validation = selection_validation_df[TARGET_COLUMNS].copy()
y_test = test_df[TARGET_COLUMNS].copy()

print(f'Selection-training rows (Knesset 21-23): {len(selection_train_df)}')
print(f'Selection-validation rows (Knesset 24): {len(selection_validation_df)}')
print(f'Final-training rows (Knesset 21-24): {len(final_train_df)}')
print(f'Test rows (Knesset 25): {len(test_df)}')


Selection-training rows (Knesset 21-23): 674
Selection-validation rows (Knesset 24): 224
Final-training rows (Knesset 21-24): 898
Test rows (Knesset 25): 226


## 4. Reusable modeling functions



In [5]:
def prepare_feature_matrices(
    training_data,
    validation_data,
    feature_columns
):
    """Fit median imputation on training data only."""

    usable_features = [
        column
        for column in feature_columns
        if not training_data[column].isna().all()
    ]

    if not usable_features:
        raise ValueError("No usable features remain.")

    imputer = SimpleImputer(strategy="median")

    X_train = pd.DataFrame(
        imputer.fit_transform(training_data[usable_features]),
        columns=usable_features,
        index=training_data.index
    )

    X_validation = pd.DataFrame(
        imputer.transform(validation_data[usable_features]),
        columns=usable_features,
        index=validation_data.index
    )

    return {
        "X_train": X_train,
        "X_validation": X_validation,
        "imputer": imputer,
        "features": usable_features,
    }


def evaluate_predictions(y_true, predictions, model_name):
    """Return overall and target-level MAE and R2."""

    y_true_array = np.asarray(y_true, dtype=float)
    predictions = np.asarray(predictions, dtype=float)

    overall_result = {
        "Model": model_name,
        "Overall_MAE": float(
            mean_absolute_error(y_true_array, predictions)
        ),
        "Overall_R2": float(
            r2_score(y_true_array, predictions)
        ),
    }

    target_mae = mean_absolute_error(
        y_true_array,
        predictions,
        multioutput="raw_values"
    )

    target_r2 = r2_score(
        y_true_array,
        predictions,
        multioutput="raw_values"
    )

    target_rows = [
        {
            "Model": model_name,
            "Target": target,
            "MAE": float(mae_value),
            "R2": float(r2_value),
        }
        for target, mae_value, r2_value in zip(
            TARGET_COLUMNS,
            target_mae,
            target_r2
        )
    ]

    return overall_result, target_rows


def normalize_predictions(predictions):
    """Clip negative values and normalize each row to 100%."""

    clipped = np.clip(
        np.asarray(predictions, dtype=float),
        0,
        None
    )

    row_sums = clipped.sum(axis=1, keepdims=True)

    zero_rows = row_sums.squeeze() == 0

    if zero_rows.any():
        clipped[zero_rows] = 1.0
        row_sums = clipped.sum(axis=1, keepdims=True)

    return clipped / row_sums * 100.0


def clr_transform(percentages, epsilon=1e-4):
    """Transform percentage compositions into CLR values."""

    compositions = np.asarray(percentages, dtype=float) / 100.0
    compositions = np.clip(compositions, epsilon, None)
    compositions = compositions / compositions.sum(
        axis=1,
        keepdims=True
    )

    log_values = np.log(compositions)

    return log_values - log_values.mean(
        axis=1,
        keepdims=True
    )


def inverse_clr(clr_values):
    """Convert CLR values into positive percentages summing to 100%."""

    clr_values = np.asarray(clr_values, dtype=float)
    stabilized = clr_values - clr_values.max(
        axis=1,
        keepdims=True
    )

    exponentials = np.exp(stabilized)

    return (
        exponentials
        / exponentials.sum(axis=1, keepdims=True)
        * 100.0
    )


In [6]:
def make_xgb_regressor():
    """Return the common XGBoost configuration."""

    return XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        objective="reg:squarederror",
        n_jobs=-1,
    )


def fit_independent_model(
    training_data,
    validation_data,
    feature_columns,
    y_training,
    estimator
):
    """Train independent regressors on raw percentage targets."""

    prepared = prepare_feature_matrices(
        training_data,
        validation_data,
        feature_columns
    )

    model = MultiOutputRegressor(clone(estimator))

    model.fit(prepared["X_train"], y_training)

    predictions = model.predict(
        prepared["X_validation"]
    )

    return {
        **prepared,
        "model": model,
        "predictions": predictions,
    }


def fit_clr_model(
    training_data,
    validation_data,
    feature_columns,
    y_training
):
    """Train XGBoost models on CLR-transformed targets."""

    prepared = prepare_feature_matrices(
        training_data,
        validation_data,
        feature_columns
    )

    y_training_clr = clr_transform(y_training)

    model = MultiOutputRegressor(
        make_xgb_regressor()
    )

    model.fit(
        prepared["X_train"],
        y_training_clr
    )

    predicted_clr = model.predict(
        prepared["X_validation"]
    )

    predictions = inverse_clr(predicted_clr)

    return {
        **prepared,
        "model": model,
        "predictions": predictions,
        "predicted_clr": predicted_clr,
    }


def fit_segmented_clr_model(
    training_data,
    validation_data,
    arab_feature_columns,
    non_arab_feature_columns
):
    """Train separate CLR models for Arab and non-Arab localities."""

    group_column = "type_Arab/Non-Jewish"

    arab_train = training_data.loc[
        training_data[group_column].eq(1)
    ].copy()

    arab_validation = validation_data.loc[
        validation_data[group_column].eq(1)
    ].copy()

    non_arab_train = training_data.loc[
        training_data[group_column].eq(0)
    ].copy()

    non_arab_validation = validation_data.loc[
        validation_data[group_column].eq(0)
    ].copy()

    for name, subset in {
        "Arab training": arab_train,
        "Arab validation": arab_validation,
        "Non-Arab training": non_arab_train,
        "Non-Arab validation": non_arab_validation,
    }.items():
        if subset.empty:
            raise ValueError(f"{name} subset is empty.")

    arab_result = fit_clr_model(
        arab_train,
        arab_validation,
        arab_feature_columns,
        arab_train[TARGET_COLUMNS]
    )

    non_arab_result = fit_clr_model(
        non_arab_train,
        non_arab_validation,
        non_arab_feature_columns,
        non_arab_train[TARGET_COLUMNS]
    )

    predictions_df = pd.DataFrame(
        index=validation_data.index,
        columns=TARGET_COLUMNS,
        dtype=float
    )

    predictions_df.loc[
        arab_validation.index,
        TARGET_COLUMNS
    ] = arab_result["predictions"]

    predictions_df.loc[
        non_arab_validation.index,
        TARGET_COLUMNS
    ] = non_arab_result["predictions"]

    if predictions_df.isna().any().any():
        raise ValueError(
            "Some validation rows did not receive predictions."
        )

    return {
        "predictions": (
            predictions_df
            .loc[validation_data.index]
            .to_numpy(dtype=float)
        ),
        "arab_result": arab_result,
        "non_arab_result": non_arab_result,
        "arab_train_rows": len(arab_train),
        "arab_validation_rows": len(arab_validation),
        "non_arab_train_rows": len(non_arab_train),
        "non_arab_validation_rows": len(non_arab_validation),
    }


## 5. Model selection — all architectures evaluated on Knesset 24

תשע הארכיטקטורות מאומנות על כנסת 21–23 ומושוות על כנסת 24 (ולידציה). כנסת 25 אינה משתתפת בשלב הזה כלל.

In [7]:
full_classification_features = list(
    dict.fromkeys(demographic_features + BASE_CLASSIFICATION_FEATURES)
)

non_arab_features = list(
    dict.fromkeys(
        demographic_features
        + [
            column
            for column in BASE_CLASSIFICATION_FEATURES
            if column != 'type_Arab/Non-Jewish'
        ]
    )
)

arab_features_with_druze = list(
    dict.fromkeys(demographic_features + [DRUZE_FEATURE])
)


def build_normalized_xgb(train_data, eval_data):
    """XGBoost with locality types, post-hoc clipped and normalized to 100%."""
    result = fit_independent_model(
        train_data,
        eval_data,
        full_classification_features,
        train_data[TARGET_COLUMNS],
        make_xgb_regressor(),
    )
    return {**result, 'predictions': normalize_predictions(result['predictions'])}


def fit_mlp_clr_model(train_data, eval_data, feature_columns, y_training):
    """A small vanilla neural network (sklearn MLP) trained in CLR space.
    L2 (alpha) + early stopping stand in for dropout at this data scale;
    unlike the tree models, the network requires standardized inputs."""
    prepared = prepare_feature_matrices(train_data, eval_data, feature_columns)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(prepared['X_train'])
    X_eval_scaled = scaler.transform(prepared['X_validation'])

    model = MLPRegressor(
        hidden_layer_sizes=(32, 16),
        alpha=1.0,
        early_stopping=True,
        max_iter=2000,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train_scaled, clr_transform(y_training))
    predictions = inverse_clr(model.predict(X_eval_scaled))

    return {**prepared, 'model': model, 'scaler': scaler, 'predictions': predictions}


# One builder per architecture: builder(train_data, eval_data) -> result dict.
# The same builders serve both the selection phase and the final refit.
MODEL_BUILDERS = {
    'Linear Regression — Demographic': lambda tr, ev: fit_independent_model(
        tr, ev, demographic_features, tr[TARGET_COLUMNS], LinearRegression()),
    'Random Forest — Demographic': lambda tr, ev: fit_independent_model(
        tr, ev, demographic_features, tr[TARGET_COLUMNS],
        RandomForestRegressor(
            n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    'XGBoost — Demographic': lambda tr, ev: fit_independent_model(
        tr, ev, demographic_features, tr[TARGET_COLUMNS], make_xgb_regressor()),
    'XGBoost — Demographic + Locality Types': lambda tr, ev: fit_independent_model(
        tr, ev, full_classification_features, tr[TARGET_COLUMNS],
        make_xgb_regressor()),
    'XGBoost + Locality Types — Post-hoc Normalized': build_normalized_xgb,
    'CLR XGBoost — Demographic + Locality Types': lambda tr, ev: fit_clr_model(
        tr, ev, full_classification_features, tr[TARGET_COLUMNS]),
    'MLP (CLR) — Neural Baseline': lambda tr, ev: fit_mlp_clr_model(
        tr, ev, full_classification_features, tr[TARGET_COLUMNS]),
    'Segmented CLR': lambda tr, ev: fit_segmented_clr_model(
        tr, ev, demographic_features, non_arab_features),
    'Segmented CLR + Druze Indicator': lambda tr, ev: fit_segmented_clr_model(
        tr, ev, arab_features_with_druze, non_arab_features),
}

selection_results = []
selection_target_results = []

for model_name, builder in MODEL_BUILDERS.items():
    print(f'Evaluating on Knesset 24: {model_name}')
    result = builder(selection_train_df, selection_validation_df)

    overall_row, target_rows = evaluate_predictions(
        y_selection_validation,
        result['predictions'],
        model_name,
    )
    selection_results.append(overall_row)
    selection_target_results.extend(target_rows)

    print(
        f"  MAE: {overall_row['Overall_MAE']:.3f} | "
        f"R2: {overall_row['Overall_R2']:.3f}"
    )

model_comparison_df = (
    pd.DataFrame(selection_results)
    .sort_values('Overall_MAE')
    .reset_index(drop=True)
)
target_comparison_df = pd.DataFrame(selection_target_results)

display(model_comparison_df.round(3))

final_model_name = model_comparison_df.iloc[0]['Model']
print(f'Selected architecture (lowest Knesset-24 MAE): {final_model_name}')


Evaluating on Knesset 24: Linear Regression — Demographic
  MAE: 11.747 | R2: 0.529
Evaluating on Knesset 24: Random Forest — Demographic


  MAE: 5.468 | R2: 0.867
Evaluating on Knesset 24: XGBoost — Demographic


  MAE: 5.384 | R2: 0.869
Evaluating on Knesset 24: XGBoost — Demographic + Locality Types


  MAE: 5.115 | R2: 0.888
Evaluating on Knesset 24: XGBoost + Locality Types — Post-hoc Normalized


  MAE: 4.900 | R2: 0.904
Evaluating on Knesset 24: CLR XGBoost — Demographic + Locality Types


  MAE: 4.975 | R2: 0.882
Evaluating on Knesset 24: MLP (CLR) — Neural Baseline
  MAE: 6.663 | R2: 0.812
Evaluating on Knesset 24: Segmented CLR


  MAE: 4.635 | R2: 0.899
Evaluating on Knesset 24: Segmented CLR + Druze Indicator


  MAE: 4.645 | R2: 0.897


,Model,Overall_MAE,Overall_R2
0,Segmented CLR,4.635,0.899
1,Segmented CLR + Druze Indicator,4.645,0.897
2,XGBoost + Locality Types — Post-hoc Normalized,4.900,0.904
3,CLR XGBoost — Demographic + Locality Types,4.975,0.882
4,XGBoost — Demographic + Locality Types,5.115,0.888
5,XGBoost — Demographic,5.384,0.869
6,Random Forest — Demographic,5.468,0.867
7,MLP (CLR) — Neural Baseline,6.663,0.812
8,Linear Regression — Demographic,11.747,0.529


Selected architecture (lowest Knesset-24 MAE): Segmented CLR


## 6. Final model — retrained on Knesset 21–24, evaluated once on Knesset 25

הארכיטקטורה שנבחרה ננעלת, מאומנת מחדש על כל נתוני 21–24, ונמדדת **פעם אחת** על כנסת 25. זהו מדד הטסט המדווח של הפרויקט.

In [8]:
final_builder = MODEL_BUILDERS[final_model_name]

print(f'Retraining "{final_model_name}" on Knesset 21-24...')
final_result = final_builder(final_train_df, test_df)
final_predictions = final_result['predictions']

test_overall, test_target_rows = evaluate_predictions(
    y_test,
    final_predictions,
    final_model_name,
)

final_sums = final_predictions.sum(axis=1)

print(f'Minimum sum: {final_sums.min():.6f}')
print(f'Maximum sum: {final_sums.max():.6f}')
print(f'Minimum prediction: {final_predictions.min():.6f}')
print(f'Maximum prediction: {final_predictions.max():.6f}')
print(f"Test MAE (Knesset 25): {test_overall['Overall_MAE']:.3f}")
print(f"Test R2 (Knesset 25):  {test_overall['Overall_R2']:.3f}")


Retraining "Segmented CLR" on Knesset 21-24...


Minimum sum: 100.000000
Maximum sum: 100.000000
Minimum prediction: 0.008137
Maximum prediction: 96.433556
Test MAE (Knesset 25): 4.207
Test R2 (Knesset 25):  0.916


## 12. Save predictions, comparison tables and final model bundle

In [9]:
metadata_columns = [
    column
    for column in [
        "locality_symbol",
        "שם יישוב",
        "year",
        "target_election",
        "valid_votes",
        *BASE_CLASSIFICATION_FEATURES,
        DRUZE_FEATURE,
    ]
    if column in test_df.columns
]

validation_predictions_df = (
    test_df[
        metadata_columns + TARGET_COLUMNS
    ]
    .reset_index(drop=True)
    .copy()
)

for target_index, target in enumerate(TARGET_COLUMNS):
    predicted_column = f"Predicted_{target}"
    error_column = f"Absolute_Error_{target}"

    validation_predictions_df[
        predicted_column
    ] = final_predictions[:, target_index]

    validation_predictions_df[
        error_column
    ] = np.abs(
        validation_predictions_df[target]
        - validation_predictions_df[
            predicted_column
        ]
    )

validation_predictions_df[
    "Mean_Absolute_Error"
] = (
    validation_predictions_df[
        [
            f"Absolute_Error_{target}"
            for target in TARGET_COLUMNS
        ]
    ]
    .mean(axis=1)
)

validation_predictions_df[
    "Predicted_Total"
] = (
    validation_predictions_df[
        [
            f"Predicted_{target}"
            for target in TARGET_COLUMNS
        ]
    ]
    .sum(axis=1)
)

display(
    validation_predictions_df
    .sort_values(
        "Mean_Absolute_Error",
        ascending=False
    )
    .head(10)
    .round(3)
)


,locality_symbol,שם יישוב,year,target_election,valid_votes,type_Arab/Non-Jewish,type_Cities,type_Kibbutzim,type_Moshavim,type_other,type_Druze_Majority,Right_pct,Center_Left_pct,Haredi_pct,Arab_pct,Predicted_Right_pct,Absolute_Error_Right_pct,Predicted_Center_Left_pct,Absolute_Error_Center_Left_pct,Predicted_Haredi_pct,Absolute_Error_Haredi_pct,Predicted_Arab_pct,Absolute_Error_Arab_pct,Mean_Absolute_Error,Predicted_Total
73,33,בת שלמה,2022,Knesset_25,323,0,0,0,1,0,0,23.885,74.522,0.955,0.637,55.496,31.610,20.523,53.999,23.939,22.983,0.042,0.595,27.297,100.0
27,13,חצבה,2022,Knesset_25,407,0,0,0,1,0,0,12.750,84.750,1.250,1.250,52.057,39.307,33.585,51.165,14.326,13.076,0.033,1.217,26.191,100.0
39,18,שדה משה,2022,Knesset_25,322,0,0,0,1,0,0,35.987,61.146,2.866,0.000,68.203,32.215,13.616,47.530,18.168,15.302,0.013,0.013,23.765,100.0
101,4203,מסעדה,2022,Knesset_25,101,1,0,0,0,0,1,59.406,36.634,1.980,1.980,23.376,36.030,30.113,6.520,1.148,0.832,45.363,43.383,21.691,100.0
72,32,עוצם,2022,Knesset_25,383,0,0,0,1,0,0,48.031,3.150,48.819,0.000,74.573,26.542,16.680,13.531,8.729,40.089,0.017,0.017,20.045,100.0
141,525,סאג'ור,2022,Knesset_25,1704,1,0,0,0,0,1,69.055,28.538,0.763,1.644,29.699,39.356,59.170,30.632,6.275,5.512,4.856,3.212,19.678,100.0
103,4501,ע'ג'ר,2022,Knesset_25,547,1,0,0,0,0,0,10.189,57.736,13.396,18.679,49.040,38.851,25.590,32.146,6.186,7.210,19.184,0.504,19.678,100.0
26,1296,כסרא-סמיע,2022,Knesset_25,1945,1,0,0,0,0,1,13.556,79.041,0.000,7.404,47.834,34.278,45.588,33.452,2.985,2.985,3.593,3.811,18.632,100.0
104,4502,עין קנייא,2022,Knesset_25,31,1,0,0,0,0,1,32.258,61.290,6.452,0.000,63.124,30.866,36.733,24.557,0.106,6.345,0.037,0.037,15.451,100.0
219,962,טובא-זנגרייה,2022,Knesset_25,2083,1,0,0,0,0,0,4.394,5.891,0.531,89.184,19.446,15.052,15.189,9.298,0.868,0.336,64.498,24.686,12.343,100.0


In [10]:
# The bundle adapts to the winning architecture: a segmented CLR winner saves
# both segment models; any other architecture saves a single model + imputer.
final_model_bundle = {
    'model_name': final_model_name,
    'selection_validation_election': SELECTION_VALIDATION_ELECTION,
    'test_election': TEST_ELECTION,
    'target_columns': TARGET_COLUMNS,
    'clr_epsilon': 1e-4,
    'xgboost_parameters': {
        'n_estimators': 500,
        'learning_rate': 0.05,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': RANDOM_STATE,
        'objective': 'reg:squarederror',
    },
}

if 'arab_result' in final_result:
    final_model_bundle.update({
        'model_family': 'segmented_clr',
        'group_column': 'type_Arab/Non-Jewish',
        'druze_feature': DRUZE_FEATURE,
        'arab_model': final_result['arab_result']['model'],
        'arab_imputer': final_result['arab_result']['imputer'],
        'arab_features': final_result['arab_result']['features'],
        'non_arab_model': final_result['non_arab_result']['model'],
        'non_arab_imputer': final_result['non_arab_result']['imputer'],
        'non_arab_features': final_result['non_arab_result']['features'],
    })
else:
    final_model_bundle.update({
        'model_family': 'single',
        'model': final_result['model'],
        'imputer': final_result['imputer'],
        'scaler': final_result.get('scaler'),
        'features': final_result['features'],
    })

joblib.dump(final_model_bundle, FINAL_MODEL_BUNDLE_PATH)

model_comparison_df.to_csv(MODEL_COMPARISON_PATH, index=False, encoding='utf-8-sig')
target_comparison_df.to_csv(TARGET_COMPARISON_PATH, index=False, encoding='utf-8-sig')
validation_predictions_df.to_csv(VALIDATION_PREDICTIONS_PATH, index=False, encoding='utf-8-sig')

print('Artifacts saved:')
print(f'- {FINAL_MODEL_BUNDLE_PATH} (family: {final_model_bundle["model_family"]})')
print(f'- {MODEL_COMPARISON_PATH}')
print(f'- {TARGET_COMPARISON_PATH}')
print(f'- {VALIDATION_PREDICTIONS_PATH}')


Artifacts saved:
- ..\models\final_model_bundle.joblib (family: segmented_clr)
- ..\reports\model_comparison.csv
- ..\reports\model_comparison_by_target.csv
- ..\data\processed\validation_predictions_selected_model.csv


In [11]:
if final_model_bundle['model_family'] == 'segmented_clr':
    feature_count_fields = {
        'arab_model_feature_count': int(len(final_model_bundle['arab_features'])),
        'non_arab_model_feature_count': int(len(final_model_bundle['non_arab_features'])),
    }
else:
    feature_count_fields = {
        'model_feature_count': int(len(final_model_bundle['features'])),
    }

selection_winner_row = model_comparison_df.iloc[0]

modeling_summary = {
    'selected_model': final_model_name,
    'selection_validation_election': SELECTION_VALIDATION_ELECTION,
     # the metrics below refer to this election
    'validation_election': TEST_ELECTION,
    'selection_training_rows': int(len(selection_train_df)),
    'training_rows': int(len(final_train_df)),
    'validation_rows': int(len(test_df)),
    'demographic_feature_count': int(len(demographic_features)),
    **feature_count_fields,
    'selection_mae_knesset_24': float(selection_winner_row['Overall_MAE']),
    'selection_r2_knesset_24': float(selection_winner_row['Overall_R2']),
    'overall_mae': float(test_overall['Overall_MAE']),
    'overall_r2': float(test_overall['Overall_R2']),
    'target_results': {
        row['Target']: {
            'mae': float(row['MAE']),
            'r2': float(row['R2']),
        }
        for row in test_target_rows
    },
    'loss_function': 'Squared Error in CLR space',
    'prediction_constraint': 'Positive percentages summing to 100',
}

with open(MODELING_SUMMARY_PATH, 'w', encoding='utf-8') as file:
    json.dump(modeling_summary, file, ensure_ascii=False, indent=2)

print(f'Modeling summary saved: {MODELING_SUMMARY_PATH}')
print(f"Selection MAE (Knesset 24): {modeling_summary['selection_mae_knesset_24']:.3f}")
print(f"Test MAE (Knesset 25): {modeling_summary['overall_mae']:.3f}")


Modeling summary saved: ..\reports\modeling_summary.json
Selection MAE (Knesset 24): 4.635
Test MAE (Knesset 25): 4.207


## 13. Notebook summary

In this notebook we compared nine architectures — linear, tree-based, boosting, a small neural baseline, compositional (CLR) and segmented variants. The winner was chosen on the Knesset 24 validation election, retrained on Knesset 21–24, and evaluated once on the held-out Knesset 25 test election. Segmentation by Arab/Non-Jewish locality combined with the CLR transformation proved decisive.

In the next notebook, we will perform:
*   Performance Metrics: Detailed analysis of MAE and R².
*   Visualization: Actual vs. Predicted plots.
*   Benchmarking: Performance comparison by voting bloc.
*   Error Analysis: Breakdown of errors by locality type.
*   Deep Dive: Identifying the top 10 localities with the highest prediction errors.
*   Discussion: Reviewing model limitations and future work.
